# 03 — Trực quan hóa dữ liệu và xuất bản biểu đồ chuẩn hóa

**Nhóm 10 — IE313.F32.LT.CNTT.** Nguồn dữ liệu: Inside Airbnb Bangkok (`data/processed/bangkok_cleaned.csv`). Hạn nộp: 18/10/2026.

Ánh xạ bài giảng:
- **Bài 03 (Trực quan cơ bản)**: Kiến trúc Matplotlib 3 tầng, chuẩn Object-Oriented API (`fig, ax = plt.subplots()`). Đầy đủ tiêu đề, nhãn trục kèm đơn vị tính, chú giải và lưới toạ độ.
- **Bài 04 (Công cụ trực quan)**: Quy trình 5 bước thiết kế trực quan hóa chuẩn:
  1. *Làm rõ câu hỏi nghiên cứu* (Clarify analytical question)
  2. *Khám phá cấu trúc dữ liệu & phân loại biến* (Explore data)
  3. *Xác định thông điệp trọng tâm* (Identify core message)
  4. *Chọn dạng biểu đồ chuẩn hóa phù hợp* (Select chart type)
  5. *Tinh chỉnh thị giác & xuất bản đồ họa* (Refine styling & export)
- **Quy chuẩn kỹ thuật**: Xuất ảnh 300 DPI vào `reports/figures/*.png`. Lưu danh mục metadata đồ họa tại `figures_catalog.json` và `figures_catalog.csv`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "workspaces/final-project"]
PROJECT = next(
    (
        p
        for p in candidates
        if (p / "data/source_manifest.json").is_file() and (p / "src/visualization.py").is_file()
    ),
    None,
)
if PROJECT is None:
    raise RuntimeError("Chạy từ gốc đồ án hoặc thư mục notebooks.")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.data_loader import compute_sha256  # noqa: E402
from src.eda_stats import EXPECTED_BASELINE_SHA256  # noqa: E402
from src.visualization import (  # noqa: E402
    generate_figures_catalog,
    plot_neighbourhood_analysis,
    plot_price_distribution,
    plot_room_type_comparison,
    plot_size_correlation,
)

baseline_csv = PROJECT / "data/processed/bangkok_cleaned.csv"
actual_hash = compute_sha256(baseline_csv)
assert actual_hash == EXPECTED_BASELINE_SHA256, f"Hash baseline không khớp: {actual_hash}"
df = pd.read_csv(baseline_csv)
df.attrs["source_sha256"] = actual_hash

print(
    f"Xác thực baseline thành công: {df.shape[0]:,} dòng x {df.shape[1]} cột. SHA-256: {actual_hash}"
)

## 1. RQ1 — Phân bố giá

1. Câu hỏi: phân bố giá và phạm vi cực trị.
2. Dữ liệu: toàn bộ giá hữu hạn dương; n/phân vị lấy từ code và bảng RQ1.
3. Thông điệp: so sánh trung tâm và đuôi phân bố theo thống kê thực tế, không kết luận giá cực trị là lỗi.
4. Hình: histogram toàn miền (thang trục log10) và vùng ≤10.000 THB; thống kê vẫn trên giá gốc.
5. Nhãn: giá THB/đêm, n thực vẽ, số ngoài khung; số ngoại lệ IQR là tiêu chí riêng. Xem `EDA_METHODS.md`.

In [ ]:
fig1_path = PROJECT / "reports/figures/fig1_price_distribution.png"
fig1, _ = plot_price_distribution(df["price"], output_path=fig1_path)
display(fig1)
plt.close(fig1)
print(f"Đã xuất bản Figure 1 tại: {fig1_path}")

## 2. RQ2 — Giá theo loại phòng

1. Câu hỏi: so sánh phân bố giá giữa nhóm phòng.
2. Dữ liệu: mẫu đầy đủ từng nhóm, chưa lọc cực trị.
3. Thông điệp: mô tả trung vị và phân vị, không suy rộng thành tác động nhân quả.
4. Hình: boxplot và bar trung vị kèm Q1–Q3; mọi phân vị tính trước khi phóng trục.
5. Nhãn: n và số ngoài giới hạn 8.000 THB; Q1–Q3 không phải khoảng tin cậy. Toàn miền xem RQ1.

In [ ]:
fig2_path = PROJECT / "reports/figures/fig2_price_by_room_type.png"
fig2, _ = plot_room_type_comparison(df, output_path=fig2_path)
display(fig2)
plt.close(fig2)
print(f"Đã xuất bản Figure 2 tại: {fig2_path}")

## 3. RQ3 — Quy mô và giá

1. Câu hỏi: mức liên hệ tuyến tính của các biến quy mô với giá.
2. Dữ liệu: cặp hữu hạn hợp lệ riêng từng biến; r, p và n ở bảng RQ3.
3. Thông điệp: độ lớn r và p có ý nghĩa khác nhau; không quy kết nguyên nhân r thấp.
4. Hình: scatter và OLS trên cùng toàn bộ cặp hợp lệ như Pearson.
5. Nhãn: lấy mẫu điểm seed42; công bố n chọn/n vẽ/số ngoài trục 20.000 THB. Lấy mẫu chỉ phục vụ hiển thị, không fit đường trên phần giá bị cắt.

In [ ]:
fig3_path = PROJECT / "reports/figures/fig3_price_vs_size_scatter.png"
fig3, _ = plot_size_correlation(df, output_path=fig3_path, seed=42, sample_display_n=3000)
display(fig3)
plt.close(fig3)
print(f"Đã xuất bản Figure 3 tại: {fig3_path}")

## 4. RQ4 — Quận và loại phòng

1. Câu hỏi: mức giá theo khu vực trong từng loại phòng.
2. Dữ liệu: pivot trung vị/n/IQR; không gộp loại phòng để suy ra hiệu ứng vị trí.
3. Thông điệp: mô tả khác biệt và ô ít mẫu; không suy diễn trung tâm/ngoại ô khi chưa có bằng chứng địa lý.
4. Hình: heatmap tối đa 15 quận nhiều listing nhất; bar trong cùng Entire home/apt.
5. Nhãn: n mỗi ô, dấu * n<5 và ô n=0; bar có n≥20 chỉ để chọn phạm vi hiển thị, không bảo đảm tính đại diện.

In [ ]:
fig4a_path = PROJECT / "reports/figures/fig4_price_by_neighbourhood_heatmap.png"
fig4b_path = PROJECT / "reports/figures/fig4_price_top_neighbourhoods_bar.png"
fig4a, fig4b = plot_neighbourhood_analysis(
    df, output_path_heatmap=fig4a_path, output_path_bars=fig4b_path
)
display(fig4a)
display(fig4b)
plt.close(fig4a)
plt.close(fig4b)
print(f"Đã xuất bản Figure 4A tại: {fig4a_path}")
print(f"Đã xuất bản Figure 4B tại: {fig4b_path}")

## 5. Xuất bản toàn bộ biểu đồ 300 DPI và danh mục Figures Catalog

Thực thi `run_all_visualizations()` để sinh và lưu toàn bộ tệp ảnh 300 DPI kèm danh mục `figures_catalog.csv`.

In [ ]:
catalog_df = generate_figures_catalog(
    PROJECT / "reports/figures",
    df=df,
    output_catalog_path=PROJECT / "reports/figures/figures_catalog.csv",
)
generate_figures_catalog(
    PROJECT / "reports/figures",
    df=df,
    output_catalog_path=PROJECT / "reports/figures/figures_catalog.json",
)
display(catalog_df[["figure_filename", "rq", "title_vi", "chart_type", "units", "sample_size_n"]])

print("Tất cả biểu đồ chuẩn 300 DPI và catalog đã sẵn sàng tại reports/figures/.")
print("Hoàn thành Bước 5–6: Sẵn sàng bàn giao cho người dùng / reviewer.")

from src.eda_report import generate_findings  # noqa: E402

generate_findings(PROJECT)
print("Đã đồng bộ reports/eda_findings.md từ artifact máy đọc.")